In [1]:
def train_mlp_score_model(df):

    # ============================================
    # FEATURES & TARGET
    # ============================================

    features = [
        "POINTS",
        "LAPS",
        "MILLISECONDS",
        "WEATHER_cloudy",
        "OVERTAKEN_POSITIONS_TOTAL",
        "DNF_COUNT",
        "LAPMEAN",
        "PS_COUNT",
        "SC_COUNT",
        "DRIVER_ENCODED",
        "RACE_ENCODED"
    ]

    target = "SCORE"

    # ============================================
    # CLEAN DATA
    # ============================================

    model_df = df[features + [target]].copy()

    model_df = model_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    model_df = model_df.dropna()

    # ============================================
    # X & y
    # ============================================

    X = model_df[features]

    y = model_df[target].values.ravel()

    # ============================================
    # TRAIN TEST SPLIT
    # ============================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # ============================================
    # MLP PIPELINE
    # ============================================

    mlp_model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "mlp",
            MLPRegressor(
                hidden_layer_sizes=(128, 64, 32),
                activation="relu",
                solver="adam",
                alpha=0.0001,
                learning_rate="adaptive",
                max_iter=1000,
                random_state=42,
                early_stopping=True,
                validation_fraction=0.1
            )
        )
    ])

    # ============================================
    # TRAIN
    # ============================================

    mlp_model.fit(
        X_train,
        y_train
    )

    # ============================================
    # PREDICTIONS
    # ============================================

    y_pred = mlp_model.predict(
        X_test
    )

    # ============================================
    # METRICS
    # ============================================

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    # ============================================
    # RESULTS DATAFRAME
    # ============================================

    results_df = pd.DataFrame({
        "Actual_SCORE": y_test,
        "Predicted_SCORE": y_pred
    })

    # ============================================
    # RETURN EVERYTHING
    # ============================================

    return {
        "model": mlp_model,
        "features": features,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "predictions": results_df,
        "X_test": X_test,
        "y_test": y_test
    }